# BMRC Interactive Video Demo

Select a local video, preview its sampled frames, fit the two small BMRC residual heads on a labeled feature cache, and inspect the spatial, temporal, and fused scores.

BMRC is **supervised**: head fitting uses labeled real and generated clips. The video selected for scoring is not used to evaluate benchmark accuracy in this notebook. Scores are compatibility/ranking values, not probabilities or a validated real/fake decision threshold. This interactive demonstration does not replace the paper's held-out evaluation.

> Videos, feature caches, DINOv3 weights, and real-reference statistics are not included. Keep them local and follow their licenses. The demo trains its own small heads from your labeled cache each time you run the training cell.

## 1. Set up the environment and local assets

Create the repository environment and install a PyTorch build that matches your machine. Then start Jupyter from the repository root:

```bash
conda env create -f environment.yml
conda activate bmrc-video-detection
# Install PyTorch/torchvision for your CUDA version using pytorch.org/get-started/locally/
python -m pip install -r requirements.txt
jupyter lab notebooks/bmrc_interactive_demo.ipynb
```

Set paths to the DINOv3 source, pretrained checkpoint, STALL real-reference archive, and a BMRC feature cache. This notebook checks for sibling `STALL/` and `Layer-Informed-Feature-Likelihood/` folders for local convenience. For a fresh clone, set `DINO_V3_REPO_DIR`, `DINO_V3_WEIGHTS`, `STALL_PARAMS`, and `BMRC_TRAIN_CACHE` before starting Jupyter.


In [ ]:
from pathlib import Path
import os, sys, time, json, subprocess
import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Video, Markdown
import ipywidgets as widgets
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Open this notebook from the cloned BMRC repository.")
os.chdir(ROOT)

# Optional convenience paths for the local Deep/ folder arrangement.
def first_existing(*paths):
    return next((Path(x).expanduser().resolve() for x in paths if Path(x).expanduser().exists()), Path(paths[0]).expanduser())
DINO_REPO = Path(os.environ.get("DINO_V3_REPO_DIR", first_existing(ROOT.parent / "Layer-Informed-Feature-Likelihood/dinov3", ROOT.parent / "STALL/dinov3")))
DINO_WEIGHTS = Path(os.environ.get("DINO_V3_WEIGHTS", first_existing(DINO_REPO / "weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth", Path.home() / ".cache/torch/hub/checkpoints/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")))
STALL_PARAMS = Path(os.environ.get("STALL_PARAMS", first_existing(ROOT.parent / "STALL/precomputed/stall_params_vatex_dino_v3.npz", ROOT.parent / "Layer-Informed-Feature-Likelihood/precomputed/stall_params_vatex_dino_v3.npz")))
TRAIN_CACHE = Path(os.environ.get("BMRC_TRAIN_CACHE", first_existing(ROOT.parent / "STALL/cache/comgenvid_features.npz", ROOT / "data/my_run")))
VIDEO_DIR = Path(os.environ.get("BMRC_DEMO_VIDEO_DIR", first_existing(ROOT.parent / "STALL/datasets/demo_dataset", ROOT / "data/demo_videos")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repository:", ROOT)
print("Scoring device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
for name, path in [("DINOv3 source", DINO_REPO), ("DINOv3 checkpoint", DINO_WEIGHTS), ("STALL reference archive", STALL_PARAMS), ("BMRC training cache", TRAIN_CACHE), ("Demo video folder", VIDEO_DIR)]:
    print(f"{name}: {path} — {'found' if path.exists() else 'not found; set its path before continuing'}")

## 2. Choose a local video

The default input uses the first two seconds and samples 16 frames at 8 FPS. Use a local clip at least two seconds long. The video is read from its path and is not copied into the repository.

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm"}
video_candidates = sorted(p for p in VIDEO_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS) if VIDEO_DIR.is_dir() else []
default_video = str(video_candidates[0]) if video_candidates else ""
video_picker = widgets.Text(value=default_video, placeholder="/path/to/local/video.mp4", description="Video path:", layout=widgets.Layout(width="95%"))
display(video_picker)
if not video_candidates:
    print("No sample clips found. Enter a local video path above.")
else:
    print(f"Found {len(video_candidates)} local clips; the first clip is selected. You can replace its path.")

def selected_video_path():
    path = Path(video_picker.value).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Select an existing local video: {path}")
    return path

def sample_indices(path, seconds=2.0, fps_out=8, count=16):
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = n / fps if fps > 0 else 0.0
    if fps <= 0 or duration < seconds:
        raise ValueError(f"Need a video of at least {seconds:g}s; found {duration:.2f}s at {fps:.2f} FPS")
    times = np.arange(count, dtype=float) / fps_out
    return np.clip(np.rint(times * fps).astype(int), 0, n-1).tolist(), fps, duration

video_path = selected_video_path()
frame_indices, source_fps, duration = sample_indices(video_path)
print(f"{video_path.name}: {source_fps:.2f} FPS, {duration:.2f} s; sampling first 2 seconds at 8 FPS")

## 3. Preview the selected clip

Check the video before scoring. The sampled-frame grid shows the same 16 frame positions passed to the detector.

In [ ]:
def preview_video(path, indices):
    cap = cv2.VideoCapture(str(path)); frames=[]
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, frame = cap.read()
        if ok: frames.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    if not frames: raise RuntimeError("Could not decode frames from the selected video")
    cols=4; rows=int(np.ceil(len(frames)/cols)); fig, axes=plt.subplots(rows, cols, figsize=(12, 2.5*rows))
    for ax in np.asarray(axes).reshape(-1): ax.axis("off")
    for ax,(idx,frame) in zip(np.asarray(axes).reshape(-1),frames):
        ax.imshow(frame); ax.set_title(f"Frame {idx}"); ax.axis("off")
    plt.tight_layout(); plt.show()
    # Make a small browser-friendly preview of the first two seconds.
    preview_path=Path("/tmp") / f"{path.stem}_bmrc_preview.mp4"
    subprocess.run(["ffmpeg","-y","-ss","0","-i",str(path),"-t","2","-an","-vf","scale='min(640,iw)':-2","-c:v","libx264","-crf","30","-pix_fmt","yuv420p","-movflags","+faststart",str(preview_path)],check=True,capture_output=True)
    display(Video(str(preview_path),embed=True,width=640))
preview_video(video_path, frame_indices)

## 4. Fit the BMRC heads from labeled examples

BMRC needs labeled real and generated videos to fit its branchwise MSE residual heads. This cell accepts either (a) a prepared per-video cache with `records/` and `features/`, built by `src/prepare_cache.py`, or (b) a local feature archive with the arrays used below (the local ComGenVid feature archive has these fields). It uses a stratified train/validation split for this interactive demo. **This split is only for fitting a demo model; it is not the paper's leave-one-generator-out evaluation.**

To build the prepared cache, create a manifest with labeled real and generated training clips and run this from the repository root:

```bash
python src/prepare_cache.py manifests/my_manifest.csv \
  --output data/demo_training \
  --params "$STALL_PARAMS" \
  --dino-repo "$DINO_V3_REPO_DIR" \
  --dino-weights "$DINO_V3_WEIGHTS" \
  --device cuda
```

Then set `BMRC_TRAIN_CACHE=/path/to/data/demo_training` before starting the notebook. Alternatively, set it to a compatible local `.npz` feature archive. Do not upload videos, weights, reference statistics, or feature caches unless their licenses permit it.


In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from evidential import fit_model

# load_rows uses these environment variables when the input is a prepared BMRC cache directory.
os.environ["BMRC_REPLAY"] = str(TRAIN_CACHE)
os.environ["STALL_PARAMS"] = str(STALL_PARAMS)

def load_labeled_features(cache_path):
    cache_path = Path(cache_path)
    if cache_path.is_file() and cache_path.suffix == ".npz":
        with np.load(cache_path, allow_pickle=False) as z:
            needed={"original_features","original_percentiles","original_quality","labels"}
            missing=needed-set(z.files)
            if missing: raise ValueError(f"Feature archive is missing {sorted(missing)}")
            x=z["original_features"].astype(np.float32)
            p=z["original_percentiles"].astype(np.float32)
            q=z["original_quality"].astype(np.float32)
            y=z["labels"].astype(np.int64)
            groups=z["sources"].astype(str) if "sources" in z.files else np.array(["all"]*len(y))
        return x,p,q,y,groups
    if (cache_path/"records").is_dir() and (cache_path/"features").is_dir():
        from run_bmrc_leave_one_generator import load_rows
        data=load_rows()
        return data["features"].astype(np.float32),data["percentiles"].astype(np.float32),data["quality"].astype(np.float32),data["is_real"].astype(np.int64),data["generator"].astype(str)
    raise FileNotFoundError("Set BMRC_TRAIN_CACHE to a compatible .npz feature archive or prepared cache directory (records/ and features/).")

X_all, P_all, Q_all, y_all, groups_all = load_labeled_features(TRAIN_CACHE)
if len(np.unique(y_all)) != 2 or not np.isfinite(X_all).all():
    raise ValueError("Training cache must contain finite features from both real (label 1) and generated (label 0) videos.")
train_idx, val_idx = train_test_split(np.arange(len(y_all)), test_size=0.2, random_state=42, stratify=y_all)
print(f"Labeled examples: {len(y_all)} | train: {len(train_idx)} | validation: {len(val_idx)}")
print("Real clips:", int((y_all==1).sum()), "| generated clips:", int((y_all==0).sum()))
print("Feature-source groups:", ", ".join(sorted(set(groups_all.tolist()))))
fit_button=widgets.Button(description="Fit BMRC residual heads", button_style="primary", icon="play")
fit_output=widgets.Output()
display(fit_button,fit_output)

def fit_demo_heads(_):
    global demo_model, demo_history
    with fit_output:
        fit_output.clear_output(wait=True)
        t0=time.time()
        demo_model,demo_history=fit_model(X_all[train_idx],P_all[train_idx],Q_all[train_idx],y_all[train_idx],X_all[val_idx],P_all[val_idx],Q_all[val_idx],y_all[val_idx],seed=42,epochs=100,objective="mse")
        print(f"Heads fitted in {time.time()-t0:.1f}s; selected epoch {min(demo_history,key=lambda r:r['validation_loss'])['epoch']}.")
        print("This model is for the interactive demonstration only; no accuracy estimate is computed here.")
fit_button.on_click(fit_demo_heads)

## 5. Score the selected video

After fitting the heads, run the scoring cell. It reports the two real-reference percentiles, branch residuals, quality weights, and the unified BMRC score. A higher score means more compatible with the real-reference ranking under this demo model; it is **not** a real/fake probability.

In [ ]:
# Update these paths if you did not use the defaults in the setup cell.
if "demo_model" not in globals():
    raise RuntimeError("Click 'Fit BMRC residual heads' above first.")
if not STALL_PARAMS.is_file() or not DINO_WEIGHTS.is_file() or not DINO_REPO.is_dir():
    raise FileNotFoundError("Set valid paths for the STALL reference archive, DINOv3 source, and checkpoint.")
from stall import STALL
from evidential import FeatureExtractor
params=np.load(STALL_PARAMS,allow_pickle=False)
detector=STALL(DEVICE,params,dino_repo=str(DINO_REPO),dino_weights=str(DINO_WEIGHTS))
raw=detector.inference(str(video_path),frame_indices=frame_indices)
feature_extractor=FeatureExtractor(detector,robust=False,quantile=0.1)
branch=feature_extractor(raw["embs"])
with torch.no_grad():
    output=demo_model(torch.as_tensor(branch["features"][None,...],dtype=torch.float32))
residuals=np.array([head[0].cpu().numpy()[0] for head in output],dtype=float)
percentiles=np.asarray(branch["percentiles"],dtype=float)
quality=np.asarray(branch["quality"],dtype=float)
weights=quality/max(float(quality.sum()),1e-12)
corrected=np.clip(percentiles+residuals,0,1)
bmrc_score=float(np.sum(weights*corrected))
if not np.isfinite(raw["temp_ll"]).any():
    weights=np.array([1.0,0.0]); bmrc_score=float(corrected[0])
score_items=[
    ("Spatial real-reference percentile",float(percentiles[0])),
    ("Temporal real-reference percentile",float(percentiles[1])),
    ("Spatial MSE residual correction",float(residuals[0])),
    ("Temporal MSE residual correction",float(residuals[1])),
    ("Spatial / temporal fusion weights",f"{weights[0]:.3f} / {weights[1]:.3f}"),
    ("Reference-only score",float(np.asarray(raw["final_score"]).reshape(-1)[0])),
    ("BMRC demo score",bmrc_score),
]
result_table="| Measurement | Value |\n|:--|--:|\n"+"\n".join(f"| {k} | {v:.4f} |" if isinstance(v,(float,np.floating)) else f"| {k} | {v} |" for k,v in score_items)
display(Markdown(result_table))
fig,ax=plt.subplots(figsize=(8,3.5))
labels=["Spatial percentile","Temporal percentile","Spatial corrected","Temporal corrected","BMRC fused"]
values=[percentiles[0],percentiles[1],corrected[0],corrected[1],bmrc_score]
ax.bar(labels,values,color=["#E6A15A","#70A9D5","#F2C08A","#9CC8E8","#70AD8A"])
ax.set_ylim(0,1); ax.set_ylabel("Reference compatibility / ranking score"); ax.set_title("BMRC score components (not probabilities)"); ax.grid(axis="y",alpha=.25); plt.xticks(rotation=18,ha="right"); plt.tight_layout(); plt.show()

## Interpretation and limits

- A higher BMRC score indicates stronger compatibility with the real-reference ranking learned by the demo heads. It is not a calibrated probability.
- The MSE heads were trained with labeled real/generated examples; the complete method is supervised.
- The demo fits heads from the selected training cache and then scores a local video. It does not calculate AUC/AP or reproduce the paper tables.
- If the selected clip or its generator family appears in the training cache, the displayed score is not a held-out test result.
- Scores may shift on unfamiliar domains. A low score alone is not proof that a clip is generated.
- For the paper's generator-held-out evaluation and ablations, use the separate commands described in the repository README.

For attribution and license terms, see `licenses/STALL-CC-BY-NC-4.0.txt` and cite the STALL paper as described in the README.